# From RAGs to Agents — a Policy-Compliance Verifier

In the previous project you built a RAG system that answers questions about company
policy. In this project you'll reuse that pipeline as one tool inside a larger agentic
application: a ***verification agent*** which checks the policy compliance of employee activities on the company dashboard and proposes fixes for the violations.


An employee can take the following actions (i.e. make the following requests) on the dashboard:

<div align="center">

| Action Type | Purpose |
|---|---|
| Expense Report | Submit a business expense for reimbursement. |
| Procurement Request | Request approval to purchase goods or services from a vendor.|
| Access / Config Change | Grant access to a system or change a security configuration. |
| Time-off Request |  Request leave / time off. |
</div>

The agent's job is to watch a user perform an action on the dashboard,
check the validity of the action, finds problems, proposes fixes, and drives the UI to show the user what's wrong and how to fix them, all based on the company policy. It is a workflow that observes an action, retrieves relevant information, reasons about constraints, delegates subtasks, and acts on an application through tools.

## The workflow you will build

In [1]:
# @title Agentic Pipeline {display-mode: "form"}
from IPython.display import HTML, display
svg_code = """<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 920 140" width="100%" style="max-width:920px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="124" y="20" width="680" height="110" rx="14" fill="#f8fafc" stroke="#647084" stroke-width="1.6" stroke-dasharray="6 4"/><rect x="418" y="2" width="92" height="38" rx="8" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="464.0" y="18" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Pipeline</text><text x="464.0" y="31" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 7 🎯</text><rect x="10" y="63" width="95" height="54" rx="10" fill="#1e293b" stroke="#0f172a" stroke-width="1.6"/><text x="57.5" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#ffffff">Action 📥</text><text x="57.5" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#94a3b8">user activity</text><rect x="143" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="189.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Validate</text><text x="189.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 1 🎯</text><rect x="253" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="299.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Context</text><text x="299.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 2 🎯</text><rect x="363" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="409.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Retrieve</text><text x="409.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 3 🎯</text><rect x="473" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="519.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Verifier</text><text x="519.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 4 🎯</text><rect x="583" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="629.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Solution</text><text x="629.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 5 🎯</text><rect x="693" y="63" width="92" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="739.0" y="86" text-anchor="middle" font-size="12" font-weight="700" fill="#1f2933">Display</text><text x="739.0" y="103" text-anchor="middle" font-size="9.5" font-weight="400" fill="#647084">Part 6 🎯</text><rect x="823" y="63" width="87" height="54" rx="10" fill="#fef3c7" stroke="#d97706" stroke-width="2"/><text x="866.5" y="83" text-anchor="middle" font-size="11" font-weight="700" fill="#78350f">Feedback 📤</text><text x="866.5" y="96" text-anchor="middle" font-size="8" font-weight="500" fill="#92400e">highlight issues ·</text><text x="866.5" y="107" text-anchor="middle" font-size="8" font-weight="500" fill="#92400e">cite · suggest fix</text><path d="M105,90 L143,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M235,90 L253,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M345,90 L363,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M455,90 L473,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M565,90 L583,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M675,90 L693,90" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M785,90 L823,90" stroke="#d97706" stroke-width="1.6" fill="none" marker-end="url(#ah)"/></svg></div>"""
display(HTML(svg_code))

A reliable agentic workflow cannot operate on free-romaing text. Instead, the inputs and ouputs of an agent should be ***structured*** so that verification can be done to prevent error propagation. The agentic architecture you will build in this project is shown above, where each part handles a small step:

<div align="center">

| Component | What It Does |
|---|---|
| 1. Validate | Check whether the action is valid  |
| 2. Context | Convert the action (request) into a contextualized query to check against company policies |
| 3. Retrieve | Retrieve relevant policies to check compliance |
| 4. Verifier |  Flag all policy violations, each with a source policy chunk |
| 5. Solution | For each violation, spawn one subagent which suggests a fix to the user |
| 6. Display |  Reflect the violations and suggested fixes on the dashboard UI. |
| 7. Pipeline | Assemble the previous 6 steps into a controlled agentic workflow. |
</div>

Each 🎯 box is a function **you** implement. To keep things simple, this project does not use frameworks like LangGraph but instead guides you to construct a minimal codebase for an agentic workflow. Throughout the steps, you will explicitly pass messages as structured inputs/outputs, invoke tool calls when appropriate, and handle exceptions when errors occur. The definitions of structured data types live inside `agentic/models.py`. You are invited to refer to this file during your implementation.

## How this notebook works

You implement each function **here**, then *monkey-patch* it onto the real project code:

```python
from agentic import action_validation
def validate_action(action):
    ... # Your implementation here
action_validation.validate_action = validate_action   # your version now runs everywhere
```

The pipeline looks each function up by name at call-time, so your implementation flows
through the whole workflow and the provided tests. At the end of each part an **export
cell** writes your function to `solutions/partN_*.py` to run the full app.

## 0.1 — Setup

This notebook reads credentials from **Colab Secrets** — never paste keys into a cell. Open
the **🔑 key icon** in the left sidebar ("Secrets") and add, toggling **"Notebook access"
ON** for `OPENROUTER_API_KEY` in order to run live demo at the end.

Run the cell below. In **Colab** it clones the project, installs dependencies, and puts the
code on the import path. Running **locally inside the repo**, it just makes the repo
importable.

In [2]:
# @title Setup {display-mode: "form"}
import os
import sys
import subprocess
NOTEBOOK_DIR = os.path.abspath(".")  # captured before any chdir below
_vsc_path = globals().get("__vsc_ipynb_file__")  # VS Code Jupyter exposes the real notebook path
if _vsc_path:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(_vsc_path))

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/eth-fdd-fs26/FDD-WE6-public.git"
REPO_BRANCH = "project"
REPO_DIR = "FDD-WE6-public"


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        proc = subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        if proc.returncode != 0:
            detail = (proc.stderr or proc.stdout or "").strip()
            raise RuntimeError("git clone failed:\n" + detail)
    %pip install -q qdrant-client rank-bm25 numpy openai pypdf rich
    sys.path.insert(0, os.path.abspath(REPO_DIR))
    os.chdir(REPO_DIR)
else:
    root = NOTEBOOK_DIR
    while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, "agentic")):
        root = os.path.dirname(root)
    sys.path.insert(0, root)
    os.chdir(root)

print("Setup complete. Running in Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

Setup complete. Running in Colab: False
Working directory: c:\Users\stefa\Documents\0000_ETH\275-0005-00L AI Workshop - From Data to Solutions\Github\Project-6--Policy-Compliance-Verification-Agent


## 0.2 — OpenRouter key (Not needed until all TODOs are finished)

Everything in this notebook runs **fully offline** using small mock AI clients, so you do
not need an API key to do the project or pass the tests. The mocks return canned, structured
JSON, which is perfect for learning to *parse and orchestrate* agent output deterministically.

If you want to see the real agents reason at the end, add your
[OpenRouter](https://openrouter.ai) key as the Colab secret **`OPENROUTER_API_KEY`**.

In [3]:
from dotenv import load_dotenv

if not IN_COLAB:
    _env_path = os.path.join(root, ".env")
    print("Looking for .env at:", _env_path, "-> exists:", os.path.isfile(_env_path))
    load_dotenv(_env_path)

_k = colab_secret("OPENROUTER_API_KEY") if IN_COLAB else None
if _k:
    os.environ["OPENROUTER_API_KEY"] = _k

HAS_KEY = bool(os.environ.get("OPENROUTER_API_KEY"))
print("Live API key available:", HAS_KEY)
print("(Everything below works offline with mocks regardless.)")

Looking for .env at: c:\Users\stefa\Documents\0000_ETH\275-0005-00L AI Workshop - From Data to Solutions\Github\Project-6--Policy-Compliance-Verification-Agent\.env -> exists: True
Live API key available: True
(Everything below works offline with mocks regardless.)


## 0.3 — Imports

We import the provided scaffolding once: the data models, the action catalogue, the prompt
templates, the JSON helper, the agent classes, the RAG core, and the test suite.

In [4]:
# @title Imports {display-mode: "form"}
import inspect
import json
import pathlib

# The agentic layer (modules we will patch + the classes/models we build on):
from agentic import action_validation, context_builder, policy_tool, verifier_agent, solution_agent, display_agent
from agentic.models import (
    Action, VerificationContext, RetrievedChunk, Problem, Verdict, Solution,
    UIAction, VerificationResult,
)
from agentic.action_types import ACTION_TYPES, get_action_type, required_fields, policy_source_for
from agentic.prompts import (
    VERIFIER_SYSTEM_PROMPT, SOLUTION_SYSTEM_PROMPT,
    build_verifier_prompt, build_solution_prompt, format_action, format_policies,
)
from agentic.json_utils import extract_json
from agentic.policy_tool import PolicyRetrievalTool
from agentic.verifier_agent import VerifierAgent
from agentic.solution_agent import SolutionAgent
from agentic.pipeline import VerifierPipeline, ActionValidationError

# The RAG tool (imported — not rebuilt):
from rag.rag_core import RAGCore
from rag.models import Document
from rag.retrieval_core import SearchType

# Offline doubles + the friendly test runner:
from tests.mocks import MockEmbedder, MockLLM, ScriptedLLM
from tests.harness import TestSuite

print("Imports OK — ready to build the agent.")
print("Known action types:", list(ACTION_TYPES))

Imports OK — ready to build the agent.
Known action types: ['expense_report', 'procurement_request', 'access_change', 'time_off_request']


## 0.4 — Pretty output helpers

Run this cell to get small HTML cards for actions, retrieved policies, verdicts, solutions,
and UI actions. You'll call these throughout (`show_action`, `show_policies`, `show_verdict`,
`show_solutions`, `show_ui_actions`).

In [5]:
# @title Display helpers {display-mode: "form"}
from IPython.display import HTML, display

_INK, _MUTE, _TEAL, _AMBER, _LINE, _BG, _RED = "#1f2933", "#647084", "#0f766e", "#b45309", "#e3e8ef", "#f8fafc", "#b91c1c"
_FONT = "ui-sans-serif,system-ui,sans-serif"
_MONO = "ui-monospace,SFMono-Regular,Menlo,monospace"

def _card(inner, accent=_TEAL):
    return (f'<div style="border:1px solid {_LINE};border-left:4px solid {accent};border-radius:10px;'
            f'padding:14px 16px;margin:8px 0;background:#fff;font-family:{_FONT}">{inner}</div>')

def _clip(t, n):
    return (t[:n] + "…") if len(t) > n else t

def show_action(action):
    rows = "".join(
        f'<tr><td style="padding:3px 10px;color:{_MUTE};font-family:{_MONO};font-size:12px">{k}</td>'
        f'<td style="padding:3px 10px;color:{_INK};font-weight:600">{v}</td></tr>'
        for k, v in action.fields.items())
    head = f'<div style="font-weight:700;color:{_INK};margin-bottom:6px">🧾 {action.action_type}</div>'
    display(HTML(_card(head + f'<table style="border-collapse:collapse">{rows}</table>', _TEAL)))

def show_policies(policies, title="Retrieved policy passages"):
    items = "".join(
        f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px 10px;margin:6px 0">'
        f'<span style="font-family:{_MONO};font-size:11px;color:{_TEAL};font-weight:700">{p.source}::chunk_{p.chunk_id}</span> '
        f'<span style="color:{_TEAL};font-weight:700">{p.score:.3f}</span>'
        f'<div style="color:{_INK};font-size:13px;margin-top:2px;white-space:pre-wrap">{p.text}</div></div>'
        for p in policies)
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:4px">📚 {title} · {len(policies)}</div>{items}', _AMBER)))

def show_verdict(verdict):
    ok = verdict.is_valid
    accent = _TEAL if ok else _RED
    badge = "✅ VALID" if ok else "⛔ PROBLEMATIC"
    probs = "".join(
        f'<li style="margin:6px 0"><b style="color:{_RED}">[{p.severity}]</b> '
        f'<code style="color:{_INK}">{p.field}</code> — {p.explanation} '
        f'<span style="color:{_MUTE}">({p.policy_source}::chunk_{p.chunk_id})</span></li>'
        for p in verdict.problems)
    body = (f'<div style="font-weight:700;color:{accent}">{badge}</div>'
            f'<div style="color:{_INK};margin:4px 0 6px">{verdict.summary}</div>'
            + (f'<ul style="margin:0;padding-left:18px">{probs}</ul>' if probs else ""))
    display(HTML(_card(body, accent)))

def show_solutions(solutions):
    if not solutions:
        display(HTML(_card("<em>No solutions (action is compliant).</em>", _TEAL))); return
    items = "".join(
        f'<li style="margin:6px 0"><b style="color:{_INK}">{s.field or "—"}</b>: {s.proposed_fix} '
        f'<span style="color:{_TEAL};font-weight:700">→ {s.corrected_value!r}</span> '
        f'<span style="color:{_MUTE}">({s.policy_source}::chunk_{s.chunk_id})</span></li>'
        for s in solutions)
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:4px">🛠️ Proposed fixes</div>'
                       f'<ul style="margin:0;padding-left:18px">{items}</ul>', _AMBER)))

def show_ui_actions(ui_actions):
    icon = {"highlight": "🔴", "warning": "⚠️", "citation": "📎", "correction": "💡", "ok": "✅"}
    items = "".join(
        f'<li style="margin:4px 0"><b>{icon.get(u.type, "•")} {u.type}</b>'
        + (f' <code style="color:{_TEAL}">{u.field}</code>' if u.field else "")
        + f' — {u.message}</li>'
        for u in ui_actions)
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:4px">🖥️ UI actions (tool calls)</div>'
                       f'<ul style="margin:0;padding-left:18px;list-style:none">{items}</ul>', _TEAL)))

print("Display helpers ready: show_action, show_policies, show_verdict, show_solutions, show_ui_actions")

Display helpers ready: show_action, show_policies, show_verdict, show_solutions, show_ui_actions


---
# Part 1 — The action & the verification trigger

A reliable agent starts from a **structured input**, not a blob of text. Every dashboard
action is an `Action` object:

```python
Action(action_type="expense_report",
       fields={"category": "hotel", "amount": 420, "region": "switzerland", ...},
       context={"role": "employee", "department": "Sales"})
```

Before spending tokens on reasoning, we run cheap deterministic checks using `validate_action`: it returns a **list of error strings** (where an empty list means the action is valid).

What you need (provided):

<div align="center">

|In python code | What it gives |
|---|---|
|`ACTION_TYPE`| A dictionary which indicates for each action type the (required/optional) fields, their allowed values and the policy source (print to see). Only the action types indicated here are allowed. |
|`required_fields(action_type: str)` | A list of required fields for an action type|
</div>

### 🎯 Your turn: implement `validate_action`

In [6]:
# @title Action types and required fields {display-mode: "form"}
print(f"Available action types: {list(ACTION_TYPES)}")
print(f"Required fields per action type:\n\t", "\n\t".join([f"{k}: {required_fields(k)}" for k in ACTION_TYPES]), sep='')


Available action types: ['expense_report', 'procurement_request', 'access_change', 'time_off_request']
Required fields per action type:
	expense_report: ['category', 'amount', 'region', 'receipt_attached', 'days_since_expense']
	procurement_request: ['amount', 'num_quotes', 'approver', 'purchase_order_raised', 'vendor_name']
	access_change: ['resource_classification', 'mfa_enabled', 'shared_account', 'grant_to']
	time_off_request: ['leave_type', 'days', 'notice_days']


In [10]:
def validate_action(action):
    """Return a list of error strings for an action.
    Arguments:
        action (Action): An Action object to validate.
    Returns:
        A list of error strings. An empty list means the action is well-formed.
    """

    errors = []
    # 🎯 Step 1: If action.action_type is empty, raise the error
    #            "action_type is required" and return early
    if not action.action_type:
        errors.append("action_type is required for the input action object")
        return errors
    # 🎯 Step 2. If action.action_type not an allowed type, raise the error
    #            "unknown action_type ... (known: ...)" and return early
    if action.action_type not in ACTION_TYPES:
        known = ", ".join(sorted(ACTION_TYPES))
        errors.append(f"unknown action_type '{action.action_type}' (known: {known})")
        return errors
    # 🎯 Step 3: If action.fields is empty, raise the error
    #           "fields must not be empty" and return early
    if not action.fields:
        errors.append("fields must not be empty")
        return errors
    # 🎯 Step 4: For each required field for this action type, if it is missing or its
    #            value is None / an empty string, add an error "required field ... must not be empty"
    for field_name in required_fields(action.action_type):
        if field_name not in action.fields:
            errors.append(f"missing required field '{field_name}'")
            continue
        value = action.fields[field_name]
        if value is None or (isinstance(value, str) and value.strip() == ""):
            errors.append(f"required field '{field_name}' must not be empty")
    return errors

In [11]:
# @title Apply patching
action_validation.validate_action = validate_action
print("Patched action_validation.validate_action")

Patched action_validation.validate_action


In [12]:
# @title Test cases for validate_action {display-mode: "form"}
suite = TestSuite("Part 1 — validate_action")

def _valid():
    return Action(action_type="expense_report", fields={
        "category": "hotel", "amount": 200, "region": "switzerland",
        "receipt_attached": True, "days_since_expense": 5})

@suite.case("validate_action", "a well-formed action has no errors")
def _():
    assert validate_action(_valid()) == []

@suite.case("validate_action", "an unknown action_type is rejected")
def _():
    assert validate_action(Action(action_type="launch_rocket", fields={"x": 1}))

@suite.case("validate_action", "a missing required field is reported by name")
def _():
    a = _valid(); del a.fields["amount"]
    assert any("amount" in e for e in validate_action(a))

@suite.case("validate_action", "falsy-but-present values (0 / False) are valid")
def _():
    a = _valid(); a.fields["amount"] = 0; a.fields["receipt_attached"] = False
    assert validate_action(a) == []

suite.run()

╭──────────────────────────╮
│ Part 1 — validate_action │
╰──────────────────────────╯

validate_action  4/4

✓ a well-formed action has no errors

✓ an unknown action_type is rejected

✓ a missing required field is reported by name

✓ falsy-but-present values (0 / False) are valid

               Summary               
                                     
  Function          Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  validate_action    4/4     ✓ PASS

╭─────────────────────────────────────╮
│ All 4 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

In [13]:
# @title See how it works on an example
_demo = Action.from_dict(json.load(open("examples/expense_hotel.json")))
show_action(_demo)
print("validation errors:", validate_action(_demo), " (expected: [] since it's a valid action)")

category,hotel
amount,420
region,switzerland
receipt_attached,True
days_since_expense,6
description,One night at a hotel during a client visit in Zurich.


validation errors: []  (expected: [] since it's a valid action)


---
# Part 2 — Building the verification context

Now we turn the action into a **query for the policy database**. This is the bridge to the
RAG tool: a natural-language `query` plus a `metadata_filter` that narrows retrieval to the
one policy document governing this action type.


What you need (provided):

<div align="center">

|In python code | What it gives |
|---|---|
|`get_action_type(action_type: str)`| `ACTION_TYPES[action_type]` for allowed action type, else None |
|`policy_source_for(action_type: str)` | The governing policy filename for the action type; None if action type is not allowed|
</div>

### 🎯 Your turn: implement `build_context`

In [14]:
def build_context(action):
    """Turn an Action into a VerificationContext for retrieval.
    Arguments:
        action (Action): An Action object to build context for.
    Returns:
        A VerificationContext object containing the query, metadata filter, summary, and action ID.
    """

    # 🎯 details about the action type
    spec = get_action_type(action.action_type) 
    # name of the action type
    label = spec["label"] if spec else action.action_type 
    field_str = ", ".join(f"{k}: {v}" for k, v in action.fields.items())
    summary = f"{label} — {field_str}" if field_str else label
    # 🎯 a string that asks if the action named `label` is allowed under company policy, with details `field_str`
    query = f"Is this {label} action allowed under company policy? Details: {field_str}" 
    # 🎯 policy source for the action type
    source = policy_source_for(action.action_type)
    metadata_filter = {"source": source} if source else None
    return VerificationContext(query=query, metadata_filter=metadata_filter,
                               summary=summary, action_id=action.id)


In [15]:
# @title Apply patching
context_builder.build_context = build_context
print("Patched context_builder.build_context")

Patched context_builder.build_context


In [16]:
# @title Test cases for build_context {display-mode: "form"}
suite = TestSuite("Part 2 — build_context")

_a = Action(action_type="expense_report", fields={
    "category": "hotel", "amount": 420, "region": "switzerland",
    "receipt_attached": True, "days_since_expense": 6})

@suite.case("build_context", "the query mentions the concrete field values")
def _():
    ctx = build_context(_a); assert "420" in ctx.query and "hotel" in ctx.query

@suite.case("build_context", "the metadata filter narrows to the governing policy")
def _():
    assert build_context(_a).metadata_filter == {"source": "expense_reimbursement_policy.md"}

@suite.case("build_context", "the action id is carried into the context")
def _():
    assert build_context(_a).action_id == _a.id

suite.run()

╭────────────────────────╮
│ Part 2 — build_context │
╰────────────────────────╯

build_context  3/3

✓ the query mentions the concrete field values

✓ the metadata filter narrows to the governing policy

✓ the action id is carried into the context

              Summary              
                                   
  Function        Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  build_context    3/3     ✓ PASS

╭─────────────────────────────────────╮
│ All 3 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

In [17]:
# @title See how it works on an example
_ctx = build_context(_a)
print("Building VerificationContext from the following action with id:", _a.id)
show_action(_a)
print("The VerificationContext object contains:")
print("  query          :", _ctx.query)
print("  metadata_filter:", _ctx.metadata_filter)
print("  summary        :", _ctx.summary)
print("  action_id      :", _ctx.action_id)

Building VerificationContext from the following action with id: b6ea21cd-2b34-4e23-a46f-19c393463bdc


category,hotel
amount,420
region,switzerland
receipt_attached,True
days_since_expense,6


The VerificationContext object contains:
  query          : Is this Expense Report action allowed under company policy? Details: category: hotel, amount: 420, region: switzerland, receipt_attached: True, days_since_expense: 6
  metadata_filter: {'source': 'expense_reimbursement_policy.md'}
  summary        : Expense Report — category: hotel, amount: 420, region: switzerland, receipt_attached: True, days_since_expense: 6
  action_id      : b6ea21cd-2b34-4e23-a46f-19c393463bdc


---
# Part 3 — Policy retrieval: RAG becomes a tool

The RAG pipeline is wrapped in `PolicyRetrievalTool`. The method `PolicyRetrievalTool.rag.retrieve` is a tool that the agent calls: given a `VerificationContext`, return the most relevant policy passages as **grounding evidence**, so that the downstream decisions are auditable and citable.

A few words on chunking: Since all of our documents are in markdown format, we apply hierarchical chunking by breaking each document down into the smallest sections and prepend the parent headers. For example, the document



```markdown
# Procurement Policy
Some text here
## First section
Some more text here
### A subsection
Details here
```

would be split into the following three chunks:
<div align='center'>

| chunk 0 | chunk 1 | chunk 2 |
| --- | --- | --- |
|[Section: Procurement Policy]<br><br>Some text here | [Section: Procurement Policy > First section]<br><br>Some more text here | [Section: Procurement Policy > First section > A subsection]<br><br>Details here |
</div>

This chunking approach ensures that the structure of the documents is preserved and each chunk can be easily traced to its source.




What you need (provided):

<div align="center">

|In python code | What it gives |
|---|---|
|`self.rag.retrieve(query, ...)` | A list of `(chunk, score)` with descending scores (top 5 are retrieved by default). |
| `chunk.metadata`| A dict with the `source` and `document_title` of the chunk object. |
| `chunk.index` | The index of the chunk inside its policy document. |
</div>

### 🎯 Your turn: implement `retrieve_policies`
Here, `self` refers to the `PolicyRetrievalTool` object, and a retrieved chunk is a `Chunk` object with attributes `metadata` and `index`.

In [24]:
def retrieve_policies(self, context):
    """Retrieve policy passages for a VerificationContext.
    Arguments:
        context (VerificationContext): The context to retrieve policies for.
    Returns:
        A list of RetrievedChunk objects.
    """

    results = self.rag.retrieve(
        # 🎯 the query string from the context
        context.query, 
        # hybrid by default
        search_type=self.search_type, 
        # 🎯 the metadata filter from the context 
        metadata_filter=context.metadata_filter, 
    )
    return [
        RetrievedChunk(
            # 🎯 the source of the chunk (or "unknown" if not present)
            source=chunk.metadata.get("source", "unknown"),
            # 🎯 the index of the chunk in the source document
            chunk_id=chunk.index, 
            # 🎯 the document title of the chunk; or its source if no title is present; or "unknown" if neither is present
            document_title=chunk.metadata.get("title", chunk.metadata.get("source", "unknown")), 
            # 🎯 the text of the chunk
            text=chunk.text, 
            # 🎯 the closeness of the chunk to the query
            score=score, 
        )
        for chunk, score in results
    ]

In [25]:
# @title Apply patching
PolicyRetrievalTool.retrieve_policies = retrieve_policies
print("Patched PolicyRetrievalTool.retrieve_policies")

Patched PolicyRetrievalTool.retrieve_policies


Let's build the RAG tool over the real policy corpus (offline, with a mock embedder so no key is needed) and retrieve grounding for our expense action. The mock embedder is irrelevant here as we use keyword search (BM25). Once you finish the implementation and provide an openrouter key, the retrieval tool will use hybrid search by default.

In [26]:
# @title See how it works on an example
_rag = RAGCore(MockEmbedder([0.0, 0.0, 0.0]), llm=None)
_rag.ingest_path("rag/data")
tool = PolicyRetrievalTool(_rag, top_k=4, search_type=SearchType.KEYWORD)

_policies = tool.retrieve_policies(build_context(_a))
show_policies(_policies, title='Grounding for the hotel expense')

Connected to Qdrant (in-memory mode)
Collection 'documents' created (vector size: 3)
Inserted 10 chunks into 'documents'
Inserted 11 chunks into 'documents'
Inserted 5 chunks into 'documents'
Inserted 12 chunks into 'documents'
Inserted 13 chunks into 'documents'
Inserted 17 chunks into 'documents'
Inserted 12 chunks into 'documents'
Inserted 12 chunks into 'documents'
Inserted 5 chunks into 'documents'


In [27]:
# @title Test cases for retrieve_policies {display-mode: "form"}
suite = TestSuite("Part 3 — retrieve_policies")

@suite.case("retrieve_policies", "returns RetrievedChunk objects with a source")
def _():
    res = tool.retrieve_policies(build_context(_a))
    assert res and all(isinstance(r, RetrievedChunk) and r.source for r in res)

@suite.case("retrieve_policies", "the metadata filter restricts retrieval to one policy")
def _():
    res = tool.retrieve_policies(build_context(_a))
    assert all(r.source == "expense_reimbursement_policy.md" for r in res)

suite.run()

╭────────────────────────────╮
│ Part 3 — retrieve_policies │
╰────────────────────────────╯

retrieve_policies  2/2

✓ returns RetrievedChunk objects with a source

✓ the metadata filter restricts retrieval to one policy

                Summary                
                                       
  Function            Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  retrieve_policies    2/2     ✓ PASS

╭─────────────────────────────────────╮
│ All 2 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

---
# Part 4 — The agent verifier

Now the first place the LLM **reasons** — but its output is **structured**, not prose. The
verifier gets the action + the retrieved policies and returns a `Verdict` containing a list
of `Problem`s. Two design rules make this trustworthy:

- **Grounding:** the prompt embeds the retrieved policy *text*, and the system prompt tells
  the model to reason *only* from those excerpts.
- **Auditability:** every problem must cite the policy `source` filename(s) that justify it.


What you need (provided):

<div align="center">

|In python code | What it gives |
|---|---|
|`build_verifier_prompt(action, policies)` | the assembled prompt for the verifier |
| `VERIFIER_SYSTEM_PROMPT`| Persona of the verifier and its output JSON schema |
|`extract_json(raw_llm_output)` | digs JSON out of the model's reply|
</div>

The `VERIFIER_SYSTEM_PROMPT` is shown below:

```python
"""You are a compliance verification agent. You are given a user ACTION and a set of POLICY EXCERPTS retrieved from the company policy database. Decide whether the action complies with the policies. Reason ONLY from the provided excerpts — do not invent rules. Report one problem per field at fault with the source policy filename and the chunk_id of the supporting excerpt. If there are multiple problematic fields, report each as a separate problem. Respond with a SINGLE JSON object and nothing else, using exactly this schema:
{
  "status": "valid" | "problematic",
  "summary": "one sentence overall assessment",
  "problems": [
    {
      "field": "name of the action field at fault",
      "policy_source": "policy_filename.md",
      "chunk_id": <int: the chunk id of the excerpt in the source document>,
      "explanation": "why this violates the policy",
      "severity": "low" | "medium" | "high"
    }
  ]
}
If the action is fully compliant, set "status": "valid" and "problems": []."""
```
### 🎯 Your turn: implement `verify_action`

In [28]:
def verify_action(self, action, policies):
    """Return a Verdict for `action` grounded in `policies`.
    Arguments:
        action (Action): The action to verify.
        policies (List[RetrievedChunk]): The retrieved chunks to ground the verification in.
    Returns:
        A Verdict object containing the status (str), problems (List[Problem]), and summary (str).
    """

    # 🎯 construct the prompt for the verifier LLM
    prompt = build_verifier_prompt(action, policies)
    raw = self.llm.complete(prompt, system_prompt=VERIFIER_SYSTEM_PROMPT)

    # 🎯 parse the raw LLM output as JSON
    data = extract_json(raw)
    problems = [
       Problem(
           # 🎯 one problematic field (default: "")
           field=item.get("field", ""),
           # 🎯 the source policy filename (default: "")
           policy_source=item.get("policy_source", ""),
           # 🎯 the index of the chunk in the source document (-1 if not applicable)
           chunk_id=int(item.get("chunk_id", -1)),
           # 🎯 explanation of the why it is a violation (default: "") 
           explanation=item.get("explanation", ""),
           # 🎯 severity of the problem (default: "medium") 
           severity=item.get("severity", "medium"),
       )
       for item in data.get("problems", [])
    ]
    status = data.get("status", "problematic" if problems else "valid")
    if problems:
        status = "problematic"
    return Verdict(status=status, problems=problems, summary=data.get("summary", ""))

In [29]:
# @title Apply patching
VerifierAgent.verify_action = verify_action
print("Patched VerifierAgent.verify_action")

Patched VerifierAgent.verify_action


To test offline we use a **mock LLM** that returns a canned verdict. This lets us check two
things deterministically: that you *parse* the output into typed objects correctly, and that
the *grounding* (the policy text) actually reached the prompt.

In [30]:
# @title Test cases for verify_action {display-mode: "form"}
_verdict_reply = {
    "status": "problematic",
    "summary": "The hotel amount exceeds the Swiss nightly limit.",
    "problems": [{
        "field": "amount",
        "policy_source": "expense_reimbursement_policy.md",
        "chunk_id": 3,
        "explanation": "CHF 420 exceeds the CHF 250 nightly hotel limit for Switzerland.",
        "severity": "high",
    }],
}

suite = TestSuite("Part 4 — verify_action")

@suite.case("verify_action", "parses a problematic verdict into typed objects")
def _():
    v = VerifierAgent(MockLLM(_verdict_reply)).verify_action(_a, _policies)
    assert isinstance(v, Verdict) and v.status == "problematic" and v.problems[0].field == "amount" \
        and v.problems[0].policy_source == "expense_reimbursement_policy.md" \
        and v.problems[0].chunk_id == 3 and v.problems[0].explanation.startswith("CHF 420 exceeds")\
        and v.problems[0].severity == "high"

@suite.case("verify_action", "every problem cites a policy source")
def _():
    v = VerifierAgent(MockLLM(_verdict_reply)).verify_action(_a, _policies)
    assert all(p.policy_source for p in v.problems) and all(p.chunk_id >= 0 for p in v.problems)

@suite.case("verify_action", "the retrieved policy text reaches the LLM (grounding)")
def _():
    llm = MockLLM(_verdict_reply); VerifierAgent(llm).verify_action(_a, _policies)
    assert "Switzerland: CHF 250 per night" in llm.last_prompt

suite.run()

╭────────────────────────╮
│ Part 4 — verify_action │
╰────────────────────────╯

verify_action  3/3

✓ parses a problematic verdict into typed objects

✓ every problem cites a policy source

✓ the retrieved policy text reaches the LLM (grounding)

              Summary              
                                   
  Function        Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  verify_action    3/3     ✓ PASS

╭─────────────────────────────────────╮
│ All 3 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

In [31]:
# @title See how it works on an example
_verdict = VerifierAgent(MockLLM(_verdict_reply)).verify_action(_a, _policies)
show_verdict(_verdict)

---
# Part 5 — The solution subagent

The verifier detects the violations (problems) and separate subagents repairs them. For each problem, the pipeline spawns one subagent, namely a `SolutionAgent.propose_solution` call, whose only job is to suggest a fix for that one problem which concerns a single field at fault. It must also cite the source policy snippet supporting the fix.
This field, along with the suggested fix and supporting policy, is then highlighted in the UI (Part 6) for the user to make appropriate changes.

In [32]:
# @title Solution Subagents {display-mode: "form"}
display(HTML("""<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 800 120" width="100%" style="max-width:800px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="20" y="46" width="150" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="95.0" y="69" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Verifier agent</text><text x="95.0" y="86" text-anchor="middle" font-size="10" font-weight="400" fill="#647084">DETECT problems</text><rect x="230" y="46" width="150" height="54" rx="10" fill="#f8fafc" stroke="#334155" stroke-width="1.6"/><text x="305.0" y="69" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Problem</text><text x="305.0" y="86" text-anchor="middle" font-size="10" font-weight="400" fill="#647084">field · why · sources</text><rect x="430" y="46" width="150" height="54" rx="10" fill="#FF0000" stroke="#b45309" stroke-width="1.6"/><text x="505.0" y="69" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Solution subagent</text><text x="505.0" y="86" text-anchor="middle" font-size="10" font-weight="400" fill="#647084">REPAIR (1 per problem)</text><rect x="630" y="46" width="150" height="54" rx="10" fill="#f8fafc" stroke="#334155" stroke-width="1.6"/><text x="705.0" y="69" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Solution Proposal</text><text x="705.0" y="86" text-anchor="middle" font-size="10" font-weight="400" fill="#647084">fix · corrected value</text><path d="M170,73 L230,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M380,73 L430,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M580,73 L630,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/></svg></div>"""))

What you need (provided):

<div align="center">

|In python code | What it gives |
|---|---|
|`build_solution_prompt(action, problem, policies)` | the assembled prompt for the solution subagent |
| `SOLUTION_SYSTEM_PROMPT`| Persona of the subagent and its output JSON schema |
|`extract_json` | digs JSON out of the model's reply |
</div>

The returned `Solution` refers back to `problem.id`. Below is the `SOLUTION_SYSTEM_PROMPT`:

```python
"""You are a remediation agent. You are given a single compliance PROBLEM with an action and the POLICY EXCERPTS that may apply. Propose one concrete fix that would make the action compliant. Ground your fix in ONE excerpt and cite the source filename along with the chunk id. Respond with a SINGLE JSON object and nothing else, using this schema:
{
  "proposed_fix": "a short, concrete instruction to the user",
  "corrected_value": <the corrected field value, or null if not a single value>,
  "policy_source": "source_filename.md",
  "chunk_id": <int: the chunk_id of the excerpt in the source document>,
  "explanation": "why this fix resolves the problem"
}"""
```

### 🎯 Your turn: implement `propose_solution`

In [33]:
def propose_solution(self, action, problem, policies):
    """Propose a fix for one `problem` in an action based on `policies`.
    Arguments:
        action (Action): The action to propose a solution for.
        problem (Problem): The specific problem to address.
        policies (List[RetrievedChunk]): The retrieved policy chunks to ground the solution in.
    Returns:
        A Solution object containing the proposed fix, corrected value, supporting sources, and explanation.
    """

    # 🎯 construct the prompt for the solution subagent
    prompt = build_solution_prompt(action, problem, policies)
    raw = self.llm.complete(prompt, system_prompt=SOLUTION_SYSTEM_PROMPT)
    # 🎯 parse the raw LLM output as JSON
    data = extract_json(raw)
    # Extract the first problematic field (if any) and propose a fix for it
    target_field = problem.field if problem.field else None
    return Solution(
        problem_id=problem.problem_id,
        # 🎯 the field to fix
        field=target_field,            
        # 🎯 the proposed fix text
        proposed_fix=data.get("proposed_fix", ""),     
        # 🎯 the corrected value (None if not applicable)
        corrected_value=data.get("corrected_value", ""),  
        # 🎯 the policy source filename
        policy_source=data.get("policy_source", ""),    
        # 🎯 the chunk id of the supporting excerpt (-1 if not applicable)
        chunk_id=data.get("chunk_id", -1),         
        # 🎯 explanation of the fix ("" if not provided)
        explanation=data.get("explanation", ""),      
    )

In [34]:
# @title Apply patching
SolutionAgent.propose_solution = propose_solution
print("Patched SolutionAgent.propose_solution")

Patched SolutionAgent.propose_solution


In [35]:
# @title Test cases for propose_solution {display-mode: "form"}
_solution_reply = {
    "field": "amount",
    "proposed_fix": "Lower the hotel amount to CHF 250 (the Swiss nightly limit).",
    "corrected_value": 250,
    "policy_source": "expense_reimbursement_policy.md",
    "chunk_id": 3,
    "explanation": "CHF 250 is the maximum reimbursable nightly hotel rate in Switzerland.",
}

suite = TestSuite("Part 5 — propose_solution")
_problem = _verdict.problems[0]

@suite.case("propose_solution", "parses a Solution with a corrected value")
def _():
    s = SolutionAgent(MockLLM(_solution_reply)).propose_solution(_a, _problem, _policies)
    assert isinstance(s, Solution) and s.corrected_value == 250 and s.proposed_fix.startswith("Lower the hotel amount") \
        and s.policy_source == "expense_reimbursement_policy.md" and s.chunk_id == 3 and s.explanation.startswith("CHF 250 is the maximum")

@suite.case("propose_solution", "the solution refers back to its problem and field")
def _():
    s = SolutionAgent(MockLLM(_solution_reply)).propose_solution(_a, _problem, _policies)
    assert s.problem_id == _problem.problem_id and s.field == "amount"

@suite.case("propose_solution", "the problem + policy text reach the LLM (grounding)")
def _():
    llm = MockLLM(_solution_reply); SolutionAgent(llm).propose_solution(_a, _problem, _policies)
    assert "Switzerland: CHF 250 per night" in llm.last_prompt

suite.run()

╭───────────────────────────╮
│ Part 5 — propose_solution │
╰───────────────────────────╯

propose_solution  3/3

✓ parses a Solution with a corrected value

✓ the solution refers back to its problem and field

✓ the problem + policy text reach the LLM (grounding)

               Summary                
                                      
  Function           Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  propose_solution    3/3     ✓ PASS

╭─────────────────────────────────────╮
│ All 3 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

---
# Part 6 — The display "agent" acts on the UI

After the solution subagents are done with their job, the pipeline collects the `action`, `verdict`, retrieved `policies` and the proposed `solutions` into a `VerificationResult` object, so that the last agent can **act on the dashboard UI by calling tools**. To keep that controlled and auditable, it may only use a fixed `UITools` interface;
it never touches the page directly. The same function runs in tests (against a `RecordingUI`
that remembers calls) and in the web backend (against a `DirectiveCollector` that turns calls
into JSON for the frontend).

The tools you need (provided):

<div align="center">

| Tool | Use |
|---|---|
| `ui.mark_ok(message)` | the action is compliant |
| `ui.warn(message, severity)` | The severity of violation (low/medium/high) for whole action |
| `ui.highlight_field(field, message)` | flag a problematic field (message = reason) |
| `ui.attach_citation(field, source, snippet)` | attach a policy citation (source = filename.md) |
| `ui.suggest_correction(field, corrected_value, proposed_fix)` | offer a one-click fix to set value to corrected_value, with proposed_fix as text explanation |
</div>

This step is **deterministic** (no LLM): predictable UI behaviour is easier to review. You may of course delegate this task to an LLM, but the best practice is to enforce deterministic workflow whenever possible in the agentic architecture for reliability.

### 🎯 Your turn: implement `run_display_agent`

In [39]:
def run_display_agent(result, ui):
    """Translate a VerificationResult into controlled UI tool calls (returns None)."""

    verdict = result.verdict

    # 🎯 Case 1: The action is valid and complies with company policy
    if verdict.is_valid:
        ui.mark_ok(verdict.summary or "This action complies with company policy.")
        return

    # 🎯 Case 2: The action is not valid and violates company policy
    # Extract the highest severity level from the problems, defaulting to "medium" if none are present
    # Ranking is done using the `rank` dictionary, where "low" < "medium" < "high"
    rank = {"low": 0, "medium": 1, "high": 2}
    top = max((p.severity for p in verdict.problems), key=lambda s: rank.get(s, 1), default="medium")
    ui.warn(verdict.summary or "This action may violate company policy.", severity=top)

    # 🎯 highlight each bad field with its explanation as message
    for problem in verdict.problems:
        ui.highlight_field(problem.field, problem.explanation)
    # 🎯 suggest a correction for each solution proposal
    for solution in result.solutions:
        ui.suggest_correction(solution.field, solution.corrected_value, solution.proposed_fix)

    # Extract all citations
    citation_problems = {
        (p.field, p.policy_source, p.chunk_id) for p in verdict.problems if p.field
    }
    citation_solutions = {
        (s.field, s.policy_source, s.chunk_id) for s in result.solutions if s.field
    }
    all_citations = citation_problems.union(citation_solutions)
    all_chunks = { (p.source, p.chunk_id): p.text for p in result.policies }

    # 🎯 Attach citations to the UI for each unique (field, source, chunk_id) combination
    for field, source, chunk_id in all_citations:
        snippet = all_chunks.get((source, chunk_id), "")  # get the text of the chunk from chunk number `chunk_id` in the document `source` ("" if not found)
        ui.attach_citation(field, source, snippet)

In [40]:
# @title Apply patching
display_agent.run_display_agent = run_display_agent
print("Patched display_agent.run_display_agent")

Patched display_agent.run_display_agent


In [41]:
# @title Test cases for run_display_agent {display-mode: "form"}
from agentic.display_agent import RecordingUI, DirectiveCollector

# Assemble a full result from the pieces we built above.
_solution = SolutionAgent(MockLLM(_solution_reply)).propose_solution(_a, _problem, _policies)
_result = VerificationResult(action=_a, verdict=_verdict, solutions=[_solution], policies=_policies)

suite = TestSuite("Part 6 — run_display_agent")

@suite.case("run_display_agent", "a problem highlights the at-fault field")
def _():
    ui = RecordingUI(); run_display_agent(_result, ui)
    assert any(name == "highlight_field" and kw["field"] == "amount" for name, kw in ui.calls)

@suite.case("run_display_agent", "a citation is attached for the policy source")
def _():
    ui = RecordingUI(); run_display_agent(_result, ui)
    assert any(name == "attach_citation" and kw["source"] == "expense_reimbursement_policy.md" and "Switzerland: CHF 250 per night" in kw["snippet"] for name, kw in ui.calls)

@suite.case("run_display_agent", "the suggested correction carries the corrected value")
def _():
    ui = RecordingUI(); run_display_agent(_result, ui)
    assert any(name == "suggest_correction" and kw["corrected_value"] == 250 for name, kw in ui.calls)

@suite.case("run_display_agent", "a valid result only marks OK")
def _():
    ui = RecordingUI()
    run_display_agent(VerificationResult(action=_a, verdict=Verdict("valid", [], "ok")), ui)
    assert ui.names() == ["mark_ok"]

@suite.case("run_display_agent", "Warnings are marked with the correct severity")
def _():
    ui = RecordingUI(); run_display_agent(_result, ui)
    assert any(name == "warn" and kw["severity"] == "high" for name, kw in ui.calls)

suite.run()

╭────────────────────────────╮
│ Part 6 — run_display_agent │
╰────────────────────────────╯

run_display_agent  5/5

✓ a problem highlights the at-fault field

✓ a citation is attached for the policy source

✓ the suggested correction carries the corrected value

✓ a valid result only marks OK

✓ Warnings are marked with the correct severity

                Summary                
                                       
  Function            Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  run_display_agent    5/5     ✓ PASS

╭─────────────────────────────────────╮
│ All 5 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

In [42]:
# @title See how it works on an example
# Render the UI actions the backend would send the frontend.
_dc = DirectiveCollector(); run_display_agent(_result, _dc)
show_ui_actions(_dc.actions)

---
# Part 7 — Putting everything into one pipeline

You implemented all six steps. The provided `VerifierPipeline` wires them into one
`verify(action)` call. Because everything resolves at call-time, your patched functions are
the ones it runs.

The `VerifierPipeline` class has the following attributes:


<div align="center">

| Attribute | Type |
|---|---|
| `self.rag` | `RAGCore` object |
| `self.policy_tool` | `PolicyRetrievalTool` object |
| `self.verifier` |`VerifierAgent` object |
| `self.solver` | `SolutionAgent` object |
</div>

Your task is to implement the `VerifierPipeline.verify` method  which binds together all the components of the agentic workflow.

### 🎯 Your turn: implement `verify`

In [49]:
def verify(self, action):
    """Run the full agentic workflow for one action.
    Arguments:
        action (Action): The action to verify.
    Returns:
        A VerificationResult object containing the verdict, solutions, and UI actions.
    Exceptions:
        ActionValidationError: If the action does not pass the validation step.
    """

    # 🎯 Part 1 — validate action
    errors = action_validation.validate_action(action)
    if errors:
        raise ActionValidationError(errors)

    # 🎯 Part 2 — build the verification context.
    context = context_builder.build_context(action)

    # 🎯 Part 3 — retrieve grounding policies via the RAG tool.
    policies = self.policy_tool.retrieve_policies(context)

    # 🎯 Part 4 — the verifier agent reasons over action + policies.
    verdict: Verdict = self.verifier.verify_action(action, policies)

    result = VerificationResult(
        action=action, verdict=verdict, solutions=[], ui_actions=[], policies=policies
    )

    # 🎯 Part 5 — one solution subagent per detected problem.
    for problem in verdict.problems:
        result.solutions.append(self.solver.propose_solution(action, problem, policies))

    # Part 6 — the display agent acts on the UI through tools.
    ui = DirectiveCollector()
    display_agent.run_display_agent(result, ui)
    result.ui_actions = ui.actions

    return result

In [50]:
# @title Apply patching
VerifierPipeline.verify = verify
print("Patched VerifierPipeline.verify")

Patched VerifierPipeline.verify


Let's run the full workflow **offline** using a `ScriptedLLM` that returns
the canned verdict to the verifier and the canned solution to the subagent (it routes by
which agent is calling).

In [51]:
_scripted = ScriptedLLM(_verdict_reply, _solution_reply)
# Reuse the already-ingested store so we don't re-embed the corpus.
pipeline = VerifierPipeline(MockEmbedder([1.0, 0.0, 0.0]), _scripted, db=_rag.db)

result = pipeline.verify(_a)
show_action(result.action)
show_verdict(result.verdict)
show_solutions(result.solutions)
show_ui_actions(result.ui_actions)

category,hotel
amount,420
region,switzerland
receipt_attached,True
days_since_expense,6


### Live demo *(needs an OpenRouter key)*

With a real key, the same pipeline runs the actual embedder + LLM: real retrieval, real
reasoning, real corrections. Try editing the example or picking another file from
`examples/`.

These are the default models using used; you may change them in the source file:
<div align="center">

| Purpose | Name in OpenRouter | Source File |
| --- | --- | ---|
| Embedding | `google/gemini-embedding-001` | `./rag/clients/embedder.py` |
| Generation | `google/gemini-3-flash-preview` | `./rag/clients/llm.py`|

In [52]:
if HAS_KEY:
    from clients import build_clients
    embedder, llm = build_clients(os.environ["OPENROUTER_API_KEY"])
    live = VerifierPipeline(embedder, llm, db_path="./qdrant_data")
else:
    print("No API key set — skipping the live demo (the offline run above did some basic checks).")

Connected to Qdrant (local persistent mode at './qdrant_data')


In [53]:
if HAS_KEY:
    example_action = Action.from_dict(json.load(open("examples/procurement_ai_hardware.json")))
    show_action(example_action)
    live_result = live.verify(example_action)
    show_verdict(live_result.verdict)
    show_solutions(live_result.solutions)
    show_ui_actions(live_result.ui_actions)

amount,10000
num_quotes,1
approver,direct_manager
purchase_order_raised,False
vendor_name,Nvidia
description,Hardware for the AI team


## Export your functions

Run this to write your six functions into `solutions/`. Dropped into the repo, the app
(`main.py` / `server.py`) runs the full agentic workflow on **your** code via
`solutions.apply()`. **Run all your implementation + patch cells above first.**

In [55]:
# @title Export the solution files {display-mode: "form"}
_EXPORTS = [
    ("solutions/part1_action.py",
     "from agentic.action_types import ACTION_TYPES, required_fields\n",
     [validate_action]),
    ("solutions/part2_context.py",
     "from agentic.action_types import get_action_type, policy_source_for\n"
     "from agentic.models import VerificationContext\n",
     [build_context]),
    ("solutions/part3_retrieval.py",
     "from agentic.models import RetrievedChunk\n",
     [retrieve_policies]),
    ("solutions/part4_verifier.py",
     "from agentic.json_utils import extract_json\n"
     "from agentic.models import Problem, Verdict\n"
     "from agentic.prompts import VERIFIER_SYSTEM_PROMPT, build_verifier_prompt\n",
     [verify_action]),
    ("solutions/part5_solution.py",
     "from agentic.json_utils import extract_json\n"
     "from agentic.models import Solution\n"
     "from agentic.prompts import SOLUTION_SYSTEM_PROMPT, build_solution_prompt\n",
     [propose_solution]),
    ("solutions/part6_display.py",
     "",
     [run_display_agent]),
    ("solutions/part7_pipeline.py",
     "from agentic import action_validation, context_builder, display_agent\n"
     "from agentic.display_agent import DirectiveCollector\n"
     "from agentic.models import Action, VerificationResult, Verdict\n"
     "from agentic.pipeline import ActionValidationError\n",
     [verify]
    )
]

for path, header, funcs in _EXPORTS:
    body = ('"""Exported from the WE6 notebook. Do not edit by hand; re-export instead."""\n'
            "from __future__ import annotations\n" + header + "\n\n"
            + "\n\n".join(inspect.getsource(f) for f in funcs))
    p = pathlib.Path(path); p.parent.mkdir(exist_ok=True)
    p.write_text(body, encoding="utf-8")
    print(f"Wrote {path}")

try:
    from google.colab import files
    for path, _h, _f in _EXPORTS:
        files.download(path)
except Exception:
    pass

Wrote solutions/part1_action.py
Wrote solutions/part2_context.py
Wrote solutions/part3_retrieval.py
Wrote solutions/part4_verifier.py
Wrote solutions/part5_solution.py
Wrote solutions/part6_display.py
Wrote solutions/part7_pipeline.py


## Run the full application

The repo ships a **company dashboard** around your pipeline:

```bash
uv sync
uv run uvicorn server:app --reload          # backend (:8000)
cd frontend && npm install && npm run dev   # frontend (:5173)
```

Open the printed URL, paste your OpenRouter key, pick an action (Expense, Procurement, Access
change, Time-off), fill it in, and submit. The verifier runs, problematic fields are
highlighted, policy citations and one-click corrections appear — all driven by the UI actions
your display agent emitted. The backend prints `[solutions] applied student implementations:
...` at startup when it picks up your exported files.


## What you learned

You built an agentic application with access to a RAG as a tool. The LLM no longer
just answers — it observes an action, retrieves grounding, reasons into a *structured*
verdict, delegates repairs to subagents, and gives its user suggestions through a controlled UI
interface. That separation (detection / reasoning / correction / display), the structured
inputs and outputs, the grounding, and the auditable tool calls are exactly what makes real
agentic applications reliable rather than "magic wrappers around an LLM" or passing around free-flowing texts without validation.

# Bonus: What's Next (Optional)

This simplistic workflow is far from perfect and your company would probably not deploy this. To take this project one step further, you would perform a failure analysis on this workflow: For what actions does it fail, and why? For example, if one action contains too many policy violations, the retrieved documents (5 by default) may not cover all the chunks of interest and it might be better to perform one retrieval per problem instead of per verdict. In addition, the solution subagents currently does not see the valid input values for each field and can thus suggest fixes which do not make sense. What other issues do you spot and how do you fix them? To speed things up, you might even want to create another agent which actively tries out different actions and collect feedback for improvement.